## 1. Verificar el entorno

GPU NVIDIA e Internet deben estar habilitados en la configuración del notebook. Se requiere Linux x86_64 y CUDA compatible con el motor.

In [ ]:
import os, platform, subprocess, sys

print('Python:', sys.version.split()[0])
print('Sistema:', platform.system(), platform.machine())
print('Kaggle:', os.path.exists('/kaggle/working'))
try:
    result = subprocess.run(['nvidia-smi', '-L'], text=True, capture_output=True, timeout=12)
    print('GPU:', result.stdout.strip() or result.stderr.strip() or 'No detectada por nvidia-smi')
except (FileNotFoundError, subprocess.TimeoutExpired):
    try:
        import torch
        print('GPU CUDA:', torch.cuda.is_available(),
              torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
    except ImportError:
        print('No se detectó GPU. Activa el acelerador GPU en Kaggle antes de continuar.')
if platform.system() != 'Linux' or platform.machine() not in ('x86_64', 'AMD64'):
    raise RuntimeError('Este cuaderno solo está preparado para Linux x86_64.')
# Evitar descargar modelos si Kaggle no ha habilitado el acelerador.
try:
    import torch
    cuda_visible = torch.cuda.is_available()
except ImportError:
    cuda_visible = False
try:
    cuda_visible = cuda_visible or (subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, timeout=8).returncode == 0)
except (FileNotFoundError, subprocess.TimeoutExpired):
    pass
if not cuda_visible:
    raise RuntimeError('GPU NVIDIA/CUDA no disponible. Activa el acelerador de Kaggle antes de instalar los modelos.')


## 2. Obtener el código fuente

Clonar `thowilabs/yue2-music-lab` o actualizar el clon existente. La operación no sobrescribe directorios ajenos.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/thowilabs/yue2-music-lab.git'
PROJECT = Path('/kaggle/working/yue2-music-lab')

if PROJECT.exists():
    if not (PROJECT / '.git').is_dir():
        raise RuntimeError(f'{PROJECT} ya existe y no es un clon Git; no se sobrescribirá.')
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT)], check=True)

print('Proyecto:', PROJECT)
print('README:', (PROJECT / 'README.md').is_file())
revision = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Revisión Git descargada:', revision)

## 3. Instalar dependencias

Instalar las dependencias Python del repositorio y FFmpeg cuando no esté disponible.

In [ ]:
import shutil, subprocess, sys

if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements.txt')], check=True)
print('Dependencias Python y FFmpeg preparados.')

## 4. Instalar el motor y los modelos

El instalador descarga el binario CUDA de audio.cpp y los modelos GGUF de YuE2; verifica el binario mediante SHA-256.

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, 'scripts/install.py'], cwd=str(PROJECT), check=True)
print('Motor y pesos YuE2 listos.')

## 5. Configurar el acceso

La autenticación está habilitada por defecto. `DISABLE_AUTH=True` abre el panel a cualquiera que tenga su enlace; utilícelo únicamente en entornos de prueba.

In [ ]:
import os

GRADIO_PORT = 7875
API_PORT = 7876
GRADIO_SHARE = True        # Enlace público temporal gradio.live
DISABLE_AUTH = False      # Cambiar a True solo para pruebas temporales
GRADIO_USERNAME = 'admin'

# Para introducir una contraseña personalizada de forma privada:
# from getpass import getpass
# MY_PASSWORD = getpass('Contraseña de Gradio: ')

server_env = os.environ.copy()
server_env.update({
    'GRADIO_SERVER_PORT': str(GRADIO_PORT),
    'GRADIO_SERVER_NAME': '0.0.0.0',
    'YUE2_BACKEND_PORT': str(API_PORT),
    'GRADIO_SHARE': '1' if GRADIO_SHARE else '0',
    'GRADIO_DISABLE_AUTH': '1' if DISABLE_AUTH else '0',
    'GRADIO_USER': GRADIO_USERNAME,
})
# Si la definiste en la celda anterior:
# server_env['GRADIO_PASSWORD'] = MY_PASSWORD

print('Login:', 'DESACTIVADO (enlace no protegido)' if DISABLE_AUTH else 'activo')
print('Enlace público:', 'sí, temporal' if GRADIO_SHARE else 'no')

## 6. Iniciar el servicio

Arranca la API local y la interfaz Gradio. El registro de salida se almacena en `logs/kaggle-notebook.log`. No modifica procesos que ocupen los puertos configurados.

In [ ]:
import os, socket, subprocess, sys, time
from pathlib import Path

LOGS = PROJECT / 'logs'
LOGS.mkdir(exist_ok=True)
LOG_FILE = LOGS / 'kaggle-notebook.log'

def listening(port):
    with socket.socket() as sock:
        return sock.connect_ex(('127.0.0.1', port)) == 0

if listening(GRADIO_PORT):
    print(f'El puerto Gradio {GRADIO_PORT} ya está ocupado. No se iniciará una copia.')
else:
    log_handle = LOG_FILE.open('a', encoding='utf-8')
    music_lab_process = subprocess.Popen(
        [sys.executable, '-u', 'scripts/run.py'], cwd=str(PROJECT),
        env=server_env, stdout=log_handle, stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    log_handle.close()
    print('Music Lab iniciado. PID:', music_lab_process.pid)
print('Archivo de logs:', LOG_FILE)

## 7. Comprobar el arranque

Extraer el enlace público temporal desde el registro y consultar la salud de la API CUDA.

In [ ]:
import re, time, urllib.request

for attempt in range(45):
    content = LOG_FILE.read_text(encoding='utf-8', errors='replace') if LOG_FILE.exists() else ''
    urls = re.findall(r'https://[a-zA-Z0-9-]+\.gradio\.live', content)
    if urls:
        print('Enlace Gradio:', urls[-1])
        break
    if 'Error' in content or 'Traceback' in content:
        print('Se detectó un error en el registro:')
        print(content[-2500:])
        break
    time.sleep(2)
else:
    print('No se encontró enlace público. Últimos registros:')
    print(content[-2500:])

try:
    with urllib.request.urlopen(f'http://127.0.0.1:{API_PORT}/health', timeout=4) as res:
        print('API CUDA:', res.status)
except Exception as exc:
    print('API aún no disponible:', type(exc).__name__)
print('\nEl acceso se controla mediante el modo de autenticación seleccionado en el paso 5.')

## 8. Detener el servicio

Ejecutar la celda únicamente si desea detener el proceso iniciado por este notebook.

In [ ]:
import signal

# Descomenta estas líneas para detener solo el proceso iniciado arriba:
# if 'music_lab_process' in globals() and music_lab_process.poll() is None:
#     os.killpg(music_lab_process.pid, signal.SIGTERM)
#     print('YuE2 Music Lab detenido.')
print('Para detener YuE2 Music Lab, descomenta las líneas de esta celda y ejecútala.')